# Reproduction notebook: set `MODE` below to `archived` (released results), `replay` (recompute from the released trajectories) or `fresh` (retrain), then run all cells.

In [ ]:
import os
MODE = os.environ.get("REPRO_MODE", "archived")
SELECT_GROUPS = None
DEVICE = "auto"
WORKERS = min(4, os.cpu_count() or 1)
WORKDIR = os.environ.get("REPRO_WORKDIR")
MAKE_FIGURES = True
FONT_FAMILY = "auto"

## 0. Prepare an isolated working copy; replay mode needs `checkpoints/`, `results/archives/`, `results/archives_rev2/` and `results/paired_honest/` from the data asset.

In [ ]:
from pathlib import Path
import json, sys

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(p for p in candidates if (p / "passportfl").is_dir() and (p / "notebooks").is_dir())
sys.path.insert(0, str(ROOT / "notebooks"))
from reproduction_workflow import GROUPS, Runner, prepare_workdir, show_tables

WORK = prepare_workdir(ROOT, MODE, WORKDIR)
import torch
DEVICE = ("cuda" if torch.cuda.is_available() else "cpu") if DEVICE == "auto" else DEVICE
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA was selected but is unavailable in this kernel.")
runner = Runner(WORK, MODE, DEVICE, WORKERS, SELECT_GROUPS)
(WORK / "reproduction_environment.json").write_text(json.dumps({
    "python": sys.version, "torch": torch.__version__, "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}, indent=2),
    encoding="utf-8")
runner.save()
print("Repository:", ROOT)
print("Working copy:", WORK, "| torch:", torch.__version__, "| device:", DEVICE)
if MODE != "archived":
    previous = Path.cwd()
    try:
        os.chdir(WORK)
        sys.path.insert(0, str(WORK))
        from passportfl.data import load_dataset
        for dataset in ("cifar100", "cifar10"):
            load_dataset(dataset)
    finally:
        os.chdir(previous)

### Execution plan: CAL seeds 0-1 fit every threshold, seed 2 is DEV and seeds 3-7 are held out.

In [ ]:
from IPython.display import HTML, display
import html
rows = "".join("<tr><td>" + html.escape(j.group) + "</td><td>" + html.escape(j.name) +
               "</td><td>" + j.kind + "</td><td><code>" +
               html.escape(" ".join(map(str, [j.script, *j.args]))) + "</code></td></tr>"
               for j in runner.jobs)
display(HTML("<details><summary>Complete execution plan: " + str(len(runner.jobs)) +
             " jobs (expand to inspect every parameter)</summary><table>" + rows + "</table></details>"))

## 1. Train or replay the honest federations

In [ ]:
runner.group('archives')

## 2. Robust rankings and geometric criteria

In [ ]:
runner.group('geometry')

### Aggregate outputs under a free-rider

In [ ]:
agg = json.loads((WORK / "results" / "aggregate_impact.json").read_text(encoding="utf-8"))
rules = ["fedproto", "mean", "krum", "median", "trimmed_mean", "bulyan"]
print("Retained step along each rule's clean output: global echo / client removed (medians over seat-rounds)")
print("%-14s" % "configuration" + "".join("%15s" % r for r in rules) + "   Krum eta*ratio median (min)")
for cfg, c in agg["configs"].items():
    cells = []
    for r in rules:
        e = c["rules"].get(r, {})
        cells.append("%15s" % ("%.2f / %.2f" % (e["global_echo"]["progress_own"]["median"],
                                               e["leave_one_out"]["progress_own"]["median"]) if e else "n/a"))
    g = c["krum_guarantee"]
    eta = ("%.1f (%.1f)" % (g["eta_times_ratio"]["median"], g["eta_times_ratio_min"])
           if g["n_gt_2f_plus_2"] else "n <= 2f+2")
    print("%-14s" % cfg + "".join(cells) + "   " + eta)
d05 = agg["configs"]["cifar100_d05"]["rules"]
print("\nCIFAR-100, delta 0.5: echo displacement / removal displacement:",
      {r: round(d05[r]["global_echo"]["shift_over_loo"]["median"], 3) for r in ("median", "trimmed_mean", "bulyan")})
print("Krum outputs the zero update in", {k: v["rules"]["krum"]["global_echo"]["zero_output_frac"] for k, v in agg["configs"].items()})
cl = agg["closed_loop_reference"]["echo"]
print("Closed-loop echo under FedProto's mean: accuracy drop %.4f +- %.4f (ddof 1) over %d seeds; unattacked range %.4f"
      % (cl["mean_drop"], cl["sd_drop"], len(cl["seeds"]), agg["closed_loop_reference"]["honest_range"]))

### Inversion condition at every seat

In [ ]:
ic = json.loads((WORK / "results" / "inversion_condition.json").read_text(encoding="utf-8"))
print("Share of scored seat-rounds certified by the sufficient condition, and by the exact comparison")
print("%-14s%10s%10s%10s%10s" % ("configuration", "median", "trimmed", "Krum", "exact"))
for cfg, c in ic["configs"].items():
    cov = c["scored"]["coverage_seat_rounds"]
    krum = cov["krum_f1"] if cfg == "cohort05" else cov["krum_f2"]
    print("%-14s%10.3f%10.3f%10.3f%10.3f" % (cfg, cov["median"]["SC1"], cov["trimmed_mean"]["SC1"],
                                             krum["krum_tight"], cov["all_rules_exact"]))
print("Largest residual of the aggregation identity:",
      max(c["identity"]["identity_ls_max_abs"] for c in ic["configs"].values()))

## 3. FLTrust and FoolsGold references

In [ ]:
runner.group('references')

## 4. Ghost prover and passport measurements

In [ ]:
runner.group('identity')

### Opening-bound verifiers, frozen encoders and published validation predicates

In [ ]:
ge = json.loads((WORK / "results" / "ghost_extended.json").read_text(encoding="utf-8"))
print("Ghost acceptance (accepted / rounds):")
for row in ge["acceptance_table"]:
    acc = row.get("accepted", row.get("accepted_total"))
    tot = row.get("rounds", row.get("rounds_total"))
    print("  %-11s %-9s %-19s %-9s warm-up %d: %4d / %d" % (row["ghost"], row["verifier"], row["regime"],
                                                           row["scheme"], row["warmup"], acc, tot))
h = ge["honest_reference"]["acceptance"]
print("Honest reference, %d post-warm-up client-rounds: deployed %d, two-sided %d accepted"
      % (ge["honest_reference"]["n_rows"], h["deployed_logged"], h["two_sided"]))
print("Calibrated ghost parameters:", ge["calibration_search"]["selected"])
fe = json.loads((WORK / "results" / "frozen_encoder_prover.json").read_text(encoding="utf-8"))
for name, verdict in fe["verdicts"].items():
    print("\nfrozen encoder, %s: %s" % (name, verdict))
pp = json.loads((WORK / "results" / "published_predicates.json").read_text(encoding="utf-8"))
attacks = ["global_echo", "drift_extrapolation", "drift_reversal", "lagged_self", "peer_copy"]
print("\nAcceptance (percent) of published input-validation predicates on held-out client-rounds")
print("%-28s %7s" % ("predicate", "honest") + "".join("%21s" % a for a in attacks))
rows = list(pp["results"].items()) + [("joint " + k, v["joint_1pct"]) for k, v in pp["combined"].items()]
for name, e in rows:
    print("%-28s %7.1f" % (name, 100 * e["honest"]["rate"])
          + "".join("%21.1f" % (100 * e["attacks"][a]["rate"]) for a in attacks))

## 5. Mask attacks and class-support dependence

In [ ]:
runner.group('support')

## 6. Calibrate and compare the payload auditor

In [ ]:
runner.group('auditor')

### Operating points, peer-aware baselines and attacks outside the design envelope

In [ ]:
from decimal import Decimal, ROUND_HALF_UP
pct = lambda x, p: str((Decimal(str(x)) * 100).quantize(Decimal(1).scaleb(-p), rounding=ROUND_HALF_UP))
ast = json.loads((WORK / "results" / "auditor_statistics.json").read_text(encoding="utf-8"))
det = ast["frozen_threshold"]["HELDOUT"]["detectors"]
print("Worst-case detection %, with realized clean false alarms %, on held-out seeds 3-7")
print("%-34s" % "detector" + "".join("%15s" % ("nominal " + t) for t in ("10%", "5%", "1%", "0.5%")) + "  worst AUC")
for key in ("suspicion", "abl_full_nostruct", "ens_stateful", "tb_maha_pca_burnin_zabs",
            "ens_stateless", "ens_stateful_x_rep"):
    r = det[key]
    ops = r["operating_points"]
    print("%-34s" % r.get("label", key)[:33]
          + "".join("%15s" % ("%s (%s)" % (pct(ops[t]["worst_tpr"], 1), pct(ops[t]["clean_fpr"], 2)))
                    for t in ("0.1", "0.05", "0.01", "0.005")) + "  %.3f" % r["worst_auc"])

## 7. Reduced optimizer effort

In [ ]:
runner.group('effort')

## 8. Adaptive grids and optimizer-effort sweep

In [ ]:
runner.group('adaptive')

### Attackers with limited access to peer state

In [ ]:
la = json.loads((WORK / "results" / "limited_access_grid.json").read_text(encoding="utf-8"))
levels = la["config"]["levels"]
levels = levels.split(",") if isinstance(levels, str) else levels
print("Calibration identical to the release:", la["calibration"]["release_check"])
for defense, name in (("P", "payload auditor"), ("F", "fused maximum")):
    print("\n%s (frozen threshold %.4f)" % (name, la["calibration"]["tau"][defense]))
    print("%-12s %6s %8s %10s %9s %10s" % ("level", "beta", "AUC", "detected", "runs", "calls/rnd"))
    for level in levels:
        for beta in ("0.5", "1", "2"):
            c = la["cells"].get("%s|%s|beta=%s" % (defense, level, beta))
            if c:
                print("%-12s %6s %8.3f %6d/%-3d %5.0f/%-3d %10.1f" % (
                    level, beta, c["auc"], c["flagged_client_rounds_TEST"], c["n_test_client_rounds"],
                    c["tpr_run_TEST"] * c["n_test_runs"], c["n_test_runs"], c["oracle_calls_mean"]))
print("\nReleased fused grid reproduced by the current-information level:",
      {k: v.get("detection_identical", v) for k, v in la["released_fused_check"].items()})
chk = json.loads((WORK / "results" / "limited_access_checks.json").read_text(encoding="utf-8"))
print("Regression and information-leak checks all passed:", chk["all_passed"])

### Heterogeneous encoders and the FedTGP server

In [ ]:
print("Global echo, seeds 0-4: AUC (mean of seed-level AUCs) / share of victim-rounds ranked first")
rules = ("krum", "median_dev", "trimmed_mean_dev", "bulyan_dev")
print("%-14s" % "configuration" + "".join("%20s" % r for r in rules))
for cfg in ("hetero", "fedtgp_lam1", "fedtgp_lam10", "fedtgp_lam01"):
    fe = json.loads((WORK / "results" / f"fedtgp_echo_{cfg}.json").read_text(encoding="utf-8"))
    pooled = fe["threshold_free"]["fedtgp_echo"]["all_seeds"]["pooled"]
    print("%-14s" % cfg + "".join("%20s" % ("%.3f / %.3f" % (pooled[r]["auc_mean_over_seeds"], pooled[r]["rank1_frac"]))
                                  for r in rules))

## 9. Utility loss and computation cost

In [ ]:
runner.group('utility')

### Delayed peer information and a robust broadcast aggregate

In [ ]:
us = json.loads((WORK / "results" / "utility_lagged_summary.json").read_text(encoding="utf-8"))
print("Paired loss of honest prototype accuracy (honest minus scenario), five seeds, s.d. ddof 1")
for k, v in us["scenarios"].items():
    fr = v["flag_rate"]["mean"]
    print("  %-16s %+.4f +- %.4f   fused-maximum flags %s"
          % (k, v["drop"]["mean"], v["drop"]["sd"], "n/a" if fr is None else "%.1f%% of client-rounds" % (100 * fr)))
ur = json.loads((WORK / "results" / "utility_robust_summary.json").read_text(encoding="utf-8"))
for rule in ("median", "mean"):
    a = ur["rules"][rule]["across_seeds"]
    fr = a["free_rider"]["free_rider"]
    print("\n%s aggregation: honest prototype accuracy %.4f" % (rule, a["tail"]["honest"]["proto"]["mean"]))
    for sc in ("echo", "exclude", "honest_rep"):
        d = a["drop"][sc]
        print("  %-10s drop %+.4f +- %.4f (proto)   %+.4f +- %.4f (classifier)"
              % (sc, d["proto"]["mean"], d["proto"]["sd"], d["cls"]["mean"], d["cls"]["sd"]))
    print("  echo ranked first in every attacked round by",
          {r: fr[r]["rank1_fraction"]["min"] == 1.0 for r in ("krum", "median_dev", "trimmed_mean_dev")})
x = ur["cross_rule"]["median_minus_mean"]["proto"]["honest_tail"]
print("\nmedian minus mean, honest prototype accuracy: %+.4f +- %.4f" % (x["mean"], x["sd"]))
for dev in ("cpu", "cuda"):
    p = WORK / "results" / f"attacker_cost_{dev}.json"
    if p.exists():
        c = json.loads(p.read_text(encoding="utf-8"))
        print("timing, honest training on %s: attacker %.2f s, honest %.2f s per round, ratio %.1f (%s, %s)"
              % (dev, c["attacker_craft_seconds"]["median"], c["honest_local_train_seconds"]["median"],
                 c["honest_over_attacker"], c["hardware"]["cpu"], c["hardware"]["gpu"]))

## 10. Result tables

In [ ]:
runner.execute("export-result-tables", "analysis/export_tables.py", outputs=["tables/tables.json"])
show_tables(WORK)
print("CSV and JSON tables:", WORK / "tables")
for name in ("held_out_summary", "passport_ablation", "utility_seed_spread"):
    data = json.loads((WORK / "results" / (name + ".json")).read_text(encoding="utf-8"))
    display(HTML("<details><summary>" + name + "</summary><pre>" +
                 html.escape(json.dumps(data, indent=2)) + "</pre></details>"))

## 11. Result figures

In [ ]:
from matplotlib import font_manager
if FONT_FAMILY == "auto":
    try:
        font_manager.findfont("Times New Roman", fallback_to_default=False)
        FONT_FAMILY = "Times New Roman"
    except ValueError:
        FONT_FAMILY = "DejaVu Serif"
print("Plot font:", FONT_FAMILY)
panels = ["inversion", "condition", "selfreport", "support", "operating", "adaptive_summary"]
if MAKE_FIGURES:
    runner.execute("result-figures", "analysis/make_figures.py",
                   ["--only", *panels, "--keep-retired", "--font-family", FONT_FAMILY],
                   [f"figures/fig_{p}.pdf" for p in panels])
    from IPython.display import Image
    panel_titles = ["Figure 2a,b: inversion", "Figure 2c: geometric criteria", "Figure 3a,b: behavior and masks",
                    "Figure 3c: class support", "Figure 4: operating point", "Figure 5: adaptive evasion and utility"]
    for panel, title in zip(panels, panel_titles):
        display(HTML("<h3>" + title + "</h3>"))
        display(Image(filename=str(WORK / f"figures/fig_{panel}.png"), width=800))
else:
    runner.events.append({"name": "result-figures", "status": "not-run"})
    runner.save()

## 12. Verify exported tables and recorded values

In [ ]:
runner.execute("check-exported-results", "analysis/export_tables.py",
               ["--check", *(["--check-figures"] if MAKE_FIGURES else [])])
if MODE == "archived":
    runner.execute("recorded-value-checks", "analysis/verify_results.py")
else:
    print("Recorded-value assertions skipped: these are recomputed measurements.")

## 13. Execution provenance

In [ ]:
runner.save()
completed = [e for e in runner.events if "returncode" in e]
archived = [e for e in runner.events if e.get("status") == "archived"]
print("Executed commands:", len(completed), "| archived job records:", len(archived))
print("Failures:", [e["name"] for e in completed if e["returncode"]])
print("Tables:", WORK / "tables", "| Figures:", WORK / "figures")
print("Ledger:", WORK / "execution_manifest.json")